[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/svngoku/cybersup-deep-learning-intro/blob/main/notebooks/formateur/05_transformer_causal_corrige.ipynb)


# TP 05 — Mini-Transformer causal
**180 minutes · CPU · tâche synthétique de copie périodique**

Un motif de quatre symboles est répété trois fois. Les motifs complets sont disjoints entre partitions.
Les quatre premiers symboles sont aléatoires ; la perte globale ne doit donc pas nécessairement tendre vers zéro.
Le modèle utilise deux blocs pre-LN, deux têtes et une largeur 32. Aucun modèle externe n'est téléchargé.
Découpage : objectif 25 min, bloc 40 min, entraînement 40 min, ablation 40 min, restitution 35 min.

**VERSION FORMATEUR — réponses et points de contrôle en fin de notebook.**

## Préparation de l'environnement

Exécuter cette cellule en premier dans un runtime Python 3 sur **CPU**. Les bibliothèques déjà installées sont conservées ; seules celles qui manquent sont ajoutées. Aucun clonage, fichier voisin ou montage Drive n'est nécessaire. Enregistrer une copie du notebook avant de commencer. Si le lien vers le dépôt privé est inaccessible, importer le fichier fourni dans le pack étudiant.


In [ ]:
# Cellule autonome : les bibliothèques déjà présentes dans Colab sont conservées.
import importlib.util
import json
import os
import platform
import subprocess
import sys

DEPENDENCIES = {
    "numpy": "numpy==2.2.6",
    "sklearn": "scikit-learn==1.7.2",
    "torch": "torch==2.8.0",
    "matplotlib": "matplotlib==3.10.6",
}
missing = [pin for module, pin in DEPENDENCIES.items()
           if importlib.util.find_spec(module) is None]
if missing:
    print("Installation des bibliothèques absentes :", ", ".join(missing))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])

import matplotlib
import numpy as np
import sklearn
import torch

ENVIRONMENT = {
    "runtime": "google-colab" if "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ else "python-local-ou-ci",
    "python": platform.python_version(),
    "numpy": np.__version__,
    "scikit_learn": sklearn.__version__,
    "torch": torch.__version__,
    "matplotlib": matplotlib.__version__,
    "device": "cpu",
    "cuda_available": torch.cuda.is_available(),
}
print(json.dumps(ENVIRONMENT, indent=2, ensure_ascii=False))
print("Les cinq TP utilisent le CPU, même si un GPU est disponible.")


In [ ]:
import os, copy, math, random
import numpy as np
import torch
from torch import nn
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits, make_moons
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, ConfusionMatrixDisplay
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.set_num_threads(2)
device = torch.device('cpu')
print('Référence : CPU | PyTorch', torch.__version__, '| graine', SEED)

In [ ]:
import itertools
patterns=np.array(list(itertools.product(range(8),repeat=4)),dtype=np.int64)
ids=np.random.default_rng(SEED).permutation(len(patterns))
train_patterns=patterns[ids[:768]]; val_patterns=patterns[ids[768:896]]; test_patterns=patterns[ids[896:1024]]
assert not(set(map(tuple,train_patterns))&set(map(tuple,val_patterns)))
assert not(set(map(tuple,train_patterns))&set(map(tuple,test_patterns)))
assert not(set(map(tuple,val_patterns))&set(map(tuple,test_patterns)))
BOS,EOS=8,9
def make_sequences(p):
    return torch.tensor(np.concatenate([np.full((len(p),1),BOS),np.tile(p,(1,3)),np.full((len(p),1),EOS)],axis=1))
train_seq,val_seq,test_seq=map(make_sequences,(train_patterns,val_patterns,test_patterns))
print('Exemple :',train_seq[0].tolist())
print('Entrée :',train_seq[0,:-1].tolist(),'Cible :',train_seq[0,1:].tolist())

In [ ]:
class TinyCausalTransformer(nn.Module):
    def __init__(self,d=32,heads=2,layers=2,use_positions=True):
        super().__init__(); self.use_positions=use_positions
        self.embedding=nn.Embedding(10,d); self.position=nn.Embedding(14,d)
        layer=nn.TransformerEncoderLayer(d_model=d,nhead=heads,dim_feedforward=64,
                    dropout=.1,activation='gelu',batch_first=True,norm_first=True)
        # TransformerEncoder fournit des blocs ; le masque ci-dessous les rend causaux.
        self.blocks=nn.TransformerEncoder(layer,num_layers=layers,norm=nn.LayerNorm(d),enable_nested_tensor=False)
        self.head=nn.Linear(d,10)
    def forward(self,tokens):
        T=tokens.shape[1]; h=self.embedding(tokens)
        if self.use_positions: h=h+self.position(torch.arange(T,device=tokens.device))[None]
        forbidden=torch.triu(torch.ones(T,T,dtype=torch.bool,device=tokens.device),diagonal=1)
        return self.head(self.blocks(h,mask=forbidden))

model=TinyCausalTransformer()
model.eval(); inputs=train_seq[:2,:-1]
changed=inputs.clone();changed[:,6:]=(changed[:,6:]+1)%8
with torch.inference_mode():
    logits=model(inputs); altered=model(changed)
assert torch.allclose(logits[:,:6],altered[:,:6],atol=1e-6)
assert logits.shape==(2,13,10)
print('Paramètres :',sum(p.numel() for p in model.parameters()))

In [ ]:
def lm_eval(model,seq):
    model.eval()
    with torch.inference_mode():
        logits=model(seq[:,:-1]); targets=seq[:,1:]
        loss=nn.functional.cross_entropy(logits.reshape(-1,10),targets.reshape(-1)).item()
        copy_acc=(logits[:,4:12].argmax(-1)==targets[:,4:12]).float().mean().item()
    return loss,copy_acc

optimizer=torch.optim.AdamW(model.parameters(),lr=3e-3,weight_decay=.01)
EPOCHS=25; history=[]; best=copy.deepcopy(model.state_dict()); best_val=float('inf')
gen=torch.Generator().manual_seed(SEED)
for epoch in range(EPOCHS):
    model.train(); total=0.
    for ids_batch in torch.randperm(len(train_seq),generator=gen).split(64):
        seq=train_seq[ids_batch]; inp=seq[:,:-1]; target=seq[:,1:]
        optimizer.zero_grad(set_to_none=True)
        logits=model(inp); loss=nn.functional.cross_entropy(logits.reshape(-1,10),target.reshape(-1))
        loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(),1.0); optimizer.step()
        total+=loss.item()*len(seq)
    vl,copy_acc=lm_eval(model,val_seq);history.append((total/len(train_seq),vl,copy_acc))
    if vl<best_val: best_val=vl;best=copy.deepcopy(model.state_dict())
model.load_state_dict(best); history=np.array(history)
plt.plot(history[:,0],label='train');plt.plot(history[:,1],label='validation');plt.legend()
plt.xlabel('Époque');plt.ylabel('Entropie croisée / token');plt.show()
print('Validation (perte, exactitude de copie) :',lm_eval(model,val_seq))

In [ ]:
def generate(model,prefix,max_new=9,temperature=None):
    model.eval(); seq=torch.tensor([prefix],dtype=torch.long)
    with torch.inference_mode():
        for _ in range(max_new):
            if seq.shape[1]>=14: break
            z=model(seq)[:,-1]
            if temperature is None: nxt=z.argmax(-1,keepdim=True)
            else:
                assert temperature>0
                nxt=torch.multinomial(torch.softmax(z/temperature,dim=-1),1)
            seq=torch.cat([seq,nxt],dim=1)
            if nxt.item()==EOS: break
    return seq[0].tolist()

prefix=[BOS,1,2,3,4]
print('Continuation gloutonne :',generate(model,prefix))
print('Référence attendue :',[BOS,1,2,3,4,1,2,3,4,1,2,3,4,EOS])
loss,acc=lm_eval(model,test_seq)
RESULT={'tp':5,'test_cross_entropy':loss,'test_perplexity':math.exp(loss),'test_copy_accuracy':acc,
        'parameters':sum(p.numel() for p in model.parameters()),'epochs':EPOCHS}
print(RESULT)

## Ablation et rapport — 40 min
1. Réinitialiser et réentraîner avec `use_positions=False`, sans changer les autres réglages.
2. Comparer validation et précision sur les positions de copie. N'imposez pas une conclusion avant l'expérience.
3. Effectuer une génération avec température 0,7 et 1,3 ; fixer la graine de tirage.
4. Expliquer pourquoi retirer le masque n'est pas une amélioration admissible de la même tâche.
5. Discuter ce que ce laboratoire ne montre pas : langage naturel, factualité, longs contextes et coût à grande échelle.

## Extension : passage d'une image à des patches
La cellule suivante vérifie les formes ; elle n'entraîne pas un Vision Transformer.

In [ ]:
images=torch.randn(2,1,8,8)
patches=images.unfold(2,2,2).unfold(3,2,2).permute(0,2,3,1,4,5).reshape(2,16,4)
embedding=nn.Linear(4,32)(patches)
assert patches.shape==(2,16,4) and embedding.shape==(2,16,32)
print('Patches :',patches.shape,'Embeddings :',embedding.shape)

## Corrigé — objectif et limites
La cible doit être décalée. Le masque interdit les positions futures avant softmax ; un simple masque 0/1 additif ne suffit pas.
Les premiers quatre symboles sont tirés aléatoirement. Les positions de sortie 4 à 11 portent les huit copies prévisibles.
La précision de copie se mesure donc sur logits[:,4:12], pas sur les premiers tokens aléatoires ni uniquement sur EOS.
La perplexité exp(CE) s'applique à toute la séquence évaluée avec cette tokenisation ; elle ne se compare pas à celle d'un modèle de langage naturel.
Un résultat imparfait n'invalide pas la démonstration de causalité. Augmenter un budget doit être déclaré et décidé avec la validation.
Sans positions, certaines régularités restent exploitables grâce au contenu et au masque. L'effet réel doit être mesuré.
Le cache K/V n'est pas implémenté : chaque étape de génération recalcule le préfixe. L'extension patches vérifie seulement les dimensions.

## Exporter les résultats

Exécuter la cellule suivante après le bilan terminal pour enregistrer métriques, graine et versions dans un fichier JSON. Dans Colab, activer `DOWNLOAD_RESULTS` pour le télécharger. Sauvegarder également une copie du notebook avec les sorties et les réponses aux investigations. Les fichiers du runtime sont temporaires.


In [ ]:
from datetime import datetime, timezone
from pathlib import Path

DOWNLOAD_RESULTS = False  # @param {type:"boolean"}

def json_scalar(value):
    if isinstance(value, np.generic):
        return value.item()
    raise TypeError(f"Valeur non sérialisable : {type(value).__name__}")

result_path = Path("resultats") / f"tp_{RESULT['tp']:02d}.json"
result_path.parent.mkdir(exist_ok=True)
experiment = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "seed": SEED,
    "environment": ENVIRONMENT,
    "results": RESULT,
}
result_path.write_text(
    json.dumps(experiment, indent=2, ensure_ascii=False, default=json_scalar) + "\n",
    encoding="utf-8",
)
print("Résultats enregistrés :", result_path)

if DOWNLOAD_RESULTS:
    try:
        from google.colab import files
    except ImportError:
        print("Hors Colab, récupérer le fichier dans le dossier resultats/.")
    else:
        files.download(str(result_path))
